# ChemBreak35 Cloud Run

ChemBreak35 preserves the complete ChemBreak35 architecture. Only the primary CHCS judge is now Qwen3.6-35B-A3B (4-bit Q4_K_M, downloaded under /content). Gemini remains the fallback.

This notebook runs the two target models independently. Run the setup cells once, then run the **ChemDFM** cell and the **ChemLLM** cell separately. Large task, stage, and turn boundaries make the live run easy to follow, while numbered steps retain planner, gate, target, judge, CHCS, latency, technical-failure, early-stop, and running ASR details.

CHCS is one cumulative-context score: after each target turn, the judge scores the original task against all target responses in the active conversation. The response set resets when the target conversation restarts. CHCS values are never added or averaged.

Do not reuse a ChemBreak35 checkpoint. Use a new `RUN_ID` whenever you switch between dry and live mode. A completed target resumes without loading that model again.


In [ ]:
# Cell 2: run controls
DRY_RUN = True                 # Set False for the live experiment
RUN_ID = None                  # None creates cb35_dry_seed_34034 or cb35_live_seed_34034
SEED = 34034
TASK_LIMIT = None              # None means all 28 tasks
PROBE_TOKENIZERS = False       # Optional. True downloads and checks the selected tokenizer in preflight
PROBE_ROLES = False            # Optional. True makes safe test calls to the planner, gate, and both judges
SHOW_LIVE_TEXT = True          # Print source tasks, accepted candidates, and target responses during the run
LIVE_TEXT_LIMIT = 1500         # Maximum displayed characters for each text block

REPOSITORY_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
REPOSITORY_BRANCH = "main"
PROJECT_SUBDIR = "chembreak35"

# Judge loading only. Keep at 8 layers on GPU initially to leave space for
# the full-size ChemDFM/ChemLLM target on the same A100. Increase only after
# checking nvidia-smi free VRAM. 0 means CPU-only inference (slower).
QWEN_JUDGE_GPU_LAYERS = 8
QWEN_JUDGE_CONTEXT_TOKENS = 16384


In [ ]:
# Cell 3: locate an extracted ZIP under /content, or clone your repository
from pathlib import Path
import subprocess, zipfile

PROJECT_ROOT = Path('/content/chembreak35')
archive = Path('/content/ChemBreak35_Standalone_Run_Package_v35.0.1_clean.zip')
if not (PROJECT_ROOT / 'pyproject.toml').exists() and archive.is_file():
    with zipfile.ZipFile(archive) as z:
        z.extractall('/content')
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = Path('/content/chembreak35_repo')
    if not (REPO_ROOT / '.git').exists():
        if REPOSITORY_URL.startswith('PASTE_'):
            raise FileNotFoundError(
                'Upload and unzip ChemBreak35 under /content/chembreak35, '
                'or set REPOSITORY_URL to the repository containing chembreak35.'
            )
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    PROJECT_ROOT = REPO_ROOT / PROJECT_SUBDIR
PROJECT_ROOT = PROJECT_ROOT.resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'ChemBreak35 not found at {PROJECT_ROOT}')
# Prevent accidentally reusing a previous extraction in the same cloud runtime.
import tomllib
installed_meta = tomllib.loads((PROJECT_ROOT / 'pyproject.toml').read_text())['project']
if installed_meta.get('name') != 'chembreak35' or installed_meta.get('version') != '35.0.1':
    raise RuntimeError(
        f'Expected a clean ChemBreak35 v35.0.1 extraction; found {installed_meta}. '
        'Move/remove the stale /content/chembreak35 directory and extract the new ZIP.'
    )
legacy_dir = PROJECT_ROOT / 'src' / ('chembreak' + '34')
if legacy_dir.exists():
    raise RuntimeError(
        'A stale previous-version source directory is present in this extraction. '
        'Use a fresh /content/chembreak35 directory to avoid mixed-version imports.'
    )
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# Cell 4: install dependencies and activate this independent package
import subprocess, sys, importlib, site

def run_install(args):
    print('RUNNING:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)

run_install([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = PROJECT_ROOT / 'requirements-cloud-ml.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing dependency file: {requirements}')
run_install([sys.executable, '-m', 'pip', 'install', '-r', requirements])
run_install([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
# Editable installs update a .pth file that a running kernel may not reload.
# Activate the local src tree explicitly so this cell works without a runtime restart.
SRC_ROOT = (PROJECT_ROOT / 'src').resolve()
PACKAGE_ROOT = SRC_ROOT / 'chembreak35'
if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(f'ChemBreak35 source package not found: {PACKAGE_ROOT}')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
site.addsitedir(str(SRC_ROOT))
for module_name in list(sys.modules):
    if module_name == 'chembreak35' or module_name.startswith('chembreak35.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import chembreak35
loaded_from = Path(chembreak35.__file__).resolve()
if PACKAGE_ROOT not in loaded_from.parents:
    raise RuntimeError(f'ChemBreak35 imported from the wrong location: {loaded_from}')
print('ChemBreak35 import OK:', chembreak35.__version__, loaded_from)


In [ ]:
# Cell 5: GPU and persistent working storage
from pathlib import Path
import torch

STORAGE_ROOT = Path('/content/chembreak35_storage').resolve()
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print('STORAGE_ROOT =', STORAGE_ROOT)
print('CUDA available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
    print('bfloat16 supported =', torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown')
elif not DRY_RUN:
    raise RuntimeError('A live ChemBreak35 target run requires a CUDA GPU runtime.')


In [ ]:
# Cell 6: credentials for planner, gate, and fallback Gemini only
import os
if not DRY_RUN and not os.environ.get('GOOGLE_CLOUD_PROJECT', '').strip():
    raise RuntimeError('Set GOOGLE_CLOUD_PROJECT for Vertex AI calls.')
# OPENAI_API_KEY is NOT required: Qwen primary judge is a local 4-bit model.
print('Credential precheck complete. Live mode =', not DRY_RUN)


In [ ]:
# Cell 7: create an isolated runtime configuration
import copy, yaml

RUN_MODE = 'dry' if DRY_RUN else 'live'
resolved_run_id = RUN_ID or f'cb35_{RUN_MODE}_seed_{SEED}'
base_config = PROJECT_ROOT / 'configs' / 'config.cb35.yaml'
config = yaml.safe_load(base_config.read_text())
config['run'].update({
    'seed': int(SEED),
    'run_mode': RUN_MODE,
    'run_id': resolved_run_id,
    'dry_run': bool(DRY_RUN),
    'task_limit': TASK_LIMIT,
    'live_progress': True,
    'live_progress_show_text': bool(SHOW_LIVE_TEXT),
    'live_progress_text_limit': int(LIVE_TEXT_LIMIT),
    'prompts_path': str(PROJECT_ROOT / 'data' / 'prompts.csv'),
    'manifest_path': str(PROJECT_ROOT / 'data' / 'CB35_prompts28_manifest_v1.csv'),
    'lock_path': str(PROJECT_ROOT / 'data' / 'CB35_prompts28_lock_v1.json'),
    'output_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'runs'),
    'artifact_root': str(STORAGE_ROOT / RUN_MODE / resolved_run_id / 'task_artifacts'),
})
config['roles']['chcs_judge_llm']['model_path'] = str(STORAGE_ROOT / 'models' / 'Qwen3.6-35B-A3B-Q4_K_M.gguf')
for role_name in ('attack_planner_llm', 'intent_gate_llm', 'chcs_fallback_judge_llm'):
    config['roles'][role_name]['seed'] = int(SEED)
for target in config['targets']:
    target['cache_dir'] = str(STORAGE_ROOT / 'cache' / 'huggingface' / 'hub')
    target['offload_folder'] = str(STORAGE_ROOT / 'offload' / target['id'])

runtime_config_dir = STORAGE_ROOT / RUN_MODE / resolved_run_id
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config = runtime_config_dir / 'config.cb35.runtime.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print('RUN MODE =', RUN_MODE)
print('RUN ID =', resolved_run_id)
print('RUNTIME CONFIG =', runtime_config)


In [ ]:
# Cell 8: download Qwen3.6-35B-A3B 4-bit ONTO /content disk and start local judge
# Run once per new cloud runtime, BEFORE the live preflight or either target.
# Skip download and build in mock-only DRY_RUN mode.
import os, shutil, subprocess, time, urllib.request, json
from pathlib import Path

QWEN_REPO = 'ggml-org/Qwen3.6-35B-A3B-GGUF'
QWEN_FILE = 'Qwen3.6-35B-A3B-Q4_K_M.gguf'
QWEN_MODEL_DIR = STORAGE_ROOT / 'models'
QWEN_MODEL_DIR.mkdir(parents=True, exist_ok=True)
QWEN_LOCAL_PATH = QWEN_MODEL_DIR / QWEN_FILE
LLAMA_CPP_ROOT = STORAGE_ROOT / 'tools' / 'llama.cpp'
LLAMA_CPP_BIN = LLAMA_CPP_ROOT / 'build' / 'bin' / 'llama-server'
QWEN_LOG_PATH = STORAGE_ROOT / 'tools' / 'llama_cpp_judge.log'
QWEN_PORT = 18035

if DRY_RUN:
    print('DRY_RUN: local 4-bit Qwen model download and server skipped.')
else:
    from huggingface_hub import hf_hub_download
    import torch
    if not QWEN_LOCAL_PATH.exists() or QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        print('Downloading 4-bit GGUF directly to:', QWEN_LOCAL_PATH, flush=True)
        hf_hub_download(repo_id=QWEN_REPO, filename=QWEN_FILE,
                        local_dir=str(QWEN_MODEL_DIR))
    if QWEN_LOCAL_PATH.stat().st_size < 19_000_000_000:
        raise RuntimeError('4-bit model file is incomplete. Rerun this cell to resume.')
    print('4-bit model on /content:', QWEN_LOCAL_PATH,
          '| GB:', round(QWEN_LOCAL_PATH.stat().st_size / 10**9, 2))

    if not LLAMA_CPP_BIN.is_file():
        LLAMA_CPP_ROOT.parent.mkdir(parents=True, exist_ok=True)
        if not (LLAMA_CPP_ROOT / '.git').is_dir():
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_CPP_ROOT)], check=True)
        cmake_cmd = ['cmake', '-S', str(LLAMA_CPP_ROOT), '-B', str(LLAMA_CPP_ROOT / 'build'),
                     '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release']
        subprocess.run(cmake_cmd, check=True)
        subprocess.run(['cmake', '--build', str(LLAMA_CPP_ROOT / 'build'),
                        '--config', 'Release', '-j', '4', '--target', 'llama-server'], check=True)
    if not LLAMA_CPP_BIN.is_file():
        raise RuntimeError('llama.cpp server executable was not built')

    def qwen_ready():
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{QWEN_PORT}/health', timeout=3) as r:
                if r.status != 200:
                    return False
            with urllib.request.urlopen(f'http://127.0.0.1:{QWEN_PORT}/v1/models', timeout=3) as r:
                available = json.load(r)
            names = {str(item.get('id', '')) for item in available.get('data', [])}
            if f'{QWEN_REPO}:Q4_K_M' not in names:
                raise RuntimeError(f'Port {QWEN_PORT} is serving another model: {sorted(names)}')
            return True
        except RuntimeError:
            raise
        except Exception:
            return False

    if not qwen_ready():
        QWEN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        log = open(QWEN_LOG_PATH, 'ab', buffering=0)
        try:
            QWEN_SERVER_PROCESS = subprocess.Popen([
                str(LLAMA_CPP_BIN), '-m', str(QWEN_LOCAL_PATH),
                '--alias', f'{QWEN_REPO}:Q4_K_M',
                '--host', '127.0.0.1', '--port', str(QWEN_PORT),
                '--ctx-size', str(QWEN_JUDGE_CONTEXT_TOKENS),
                '--n-gpu-layers', str(QWEN_JUDGE_GPU_LAYERS),
                '--threads', '8', '--parallel', '1',
                '--batch-size', '256', '--ubatch-size', '128',
                '--jinja', '--reasoning', 'off',
            ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        finally:
            log.close()
        for _ in range(240):
            if qwen_ready():
                break
            if QWEN_SERVER_PROCESS.poll() is not None:
                tail = QWEN_LOG_PATH.read_text(errors='replace')[-6000:]
                raise RuntimeError('Qwen judge server failed to start:\n' + tail)
            time.sleep(2)
    if not qwen_ready():
        raise RuntimeError('Qwen judge server not ready. Check ' + str(QWEN_LOG_PATH))
    print('Local 4-bit primary judge is READY on 127.0.0.1:18035')
    print('GPU layers used by Qwen judge:', QWEN_JUDGE_GPU_LAYERS)
    print('Remember: ChemDFM and ChemLLM remain unmodified and load independently.')


In [ ]:
# Cell 9: package, dataset, and API/GPU preflight
from pprint import pprint
from chembreak35.preflight import run_preflight

preflight_report = run_preflight(
    runtime_config,
    probe_tokenizers=False,
    probe_roles=PROBE_ROLES,
    target_id=None,
)
pprint(preflight_report)


## Target 1: ChemDFM

Run the next two cells for ChemDFM. The run cell is independent of the ChemLLM run cell. It unloads ChemDFM in a `finally` block even if execution is interrupted.


In [ ]:
# Cell 11: optional ChemDFM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemDFM'))
else:
    print('ChemDFM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 12: run ChemDFM only
from chembreak35.runner import ChemBreak35Runner

chemdfm_runner = ChemBreak35Runner(runtime_config, 'ChemDFM')
try:
    chemdfm_summary = chemdfm_runner.run_all()
finally:
    chemdfm_runner.unload_target()
    chemdfm_runner.close()

print('ChemDFM FINAL:', chemdfm_summary['bounded_first_success'])


In [ ]:
# Cell 13: confirm GPU memory after ChemDFM
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU allocated after ChemDFM cleanup (GiB) =', round(torch.cuda.memory_allocated() / 1024**3, 3))
else:
    print('Dry run or no CUDA GPU.')


## Target 2: ChemLLM

Run the next two cells for ChemLLM. This cell creates its own runner and loads only ChemLLM. It does not depend on the ChemDFM runner or its in-memory state.


In [ ]:
# Cell 15: optional ChemLLM tokenizer preflight
if PROBE_TOKENIZERS:
    pprint(run_preflight(runtime_config, probe_tokenizers=True, probe_roles=False, target_id='ChemLLM'))
else:
    print('ChemLLM tokenizer probe skipped. Set PROBE_TOKENIZERS=True to enable it.')


In [ ]:
# Cell 16: run ChemLLM only
from chembreak35.runner import ChemBreak35Runner

chemllm_runner = ChemBreak35Runner(runtime_config, 'ChemLLM')
try:
    chemllm_summary = chemllm_runner.run_all()
finally:
    chemllm_runner.unload_target()
    chemllm_runner.close()

print('ChemLLM FINAL:', chemllm_summary['bounded_first_success'])


In [ ]:
# Cell 17: compare whichever targets have been completed in this session
comparison = {}
if 'chemdfm_summary' in globals():
    comparison['ChemDFM'] = chemdfm_summary['bounded_first_success']
if 'chemllm_summary' in globals():
    comparison['ChemLLM'] = chemllm_summary['bounded_first_success']
if not comparison:
    print('Run at least one target cell first.')
else:
    pprint(comparison)


In [ ]:
# Cell 18: locate public and private outputs
from pathlib import Path

for target_id in ('ChemDFM', 'ChemLLM'):
    target_root = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id
    release = target_root / 'release'
    internal = release / 'INTERNAL_DO_NOT_RELEASE'
    print(target_id)
    print('  public release:', release)
    print('  private transcript:', internal / 'cb35_full_transcripts.csv')
    print('  first-success review:', internal / 'cb35_first_success_review.csv')


In [ ]:
# Cell 19: create a public-results ZIP without private transcripts
import shutil

zip_root = STORAGE_ROOT / RUN_MODE / resolved_run_id / 'exports'
zip_root.mkdir(parents=True, exist_ok=True)
public_stage = zip_root / 'chembreak35_public_results'
if public_stage.exists():
    shutil.rmtree(public_stage)
public_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    release = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release'
    if release.exists():
        destination = public_stage / target_id
        shutil.copytree(release, destination, ignore=shutil.ignore_patterns('INTERNAL_DO_NOT_RELEASE'))
public_zip = shutil.make_archive(str(zip_root / f'chembreak35_{resolved_run_id}_public'), 'zip', public_stage)
print('PUBLIC ZIP =', public_zip)


In [ ]:
# Cell 20: create a private audit ZIP containing raw prompts and responses
import shutil

private_stage = zip_root / 'chembreak35_private_audit'
if private_stage.exists():
    shutil.rmtree(private_stage)
private_stage.mkdir(parents=True)
for target_id in ('ChemDFM', 'ChemLLM'):
    internal = Path(config['run']['output_root']) / config['run']['experiment_revision'] / target_id / 'release' / 'INTERNAL_DO_NOT_RELEASE'
    if internal.exists():
        shutil.copytree(internal, private_stage / target_id)
private_zip = shutil.make_archive(str(zip_root / f'chembreak35_{resolved_run_id}_private_audit'), 'zip', private_stage)
print('PRIVATE AUDIT ZIP =', private_zip)
